# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2
dataset using the `mlcroissant` library. Follow along to load, extract, 
analyze, and visualize data directly from a Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. In addition,
print the dataset title and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'
url = croissant_url

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print metadata name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns and their IDs.

We list the Record Sets and their Field `@id` values for clarity.

In [ ]:
# List all record sets and their fields (by @id)
record_sets = metadata.recordSet
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']}")
        fields = rs.get('field', [])
        print("  Fields:")
        for field in fields:
            print(f"    {field['@id']} (name: {field.get('name', '')})")

    # Print columns for first record set if present
    columns = record_sets[0].get('column', []) if record_sets else []
    for col in columns:
        print(f"Column: {col['@id']} (name: {col.get('name', '')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use the record set and field `@id`s provided above. The loaded data
is accessible via the `mlcroissant.Dataset.records` method.

In [ ]:
# Extract data from each record set
dataframes = {}

# Store all record set @ids
record_set_ids = []
for rs in (metadata.recordSet or []):
    record_set_ids.append(rs['@id'])

if record_set_ids:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for record set {record_set_id}: {df.columns.tolist()}")
        print(df.head())
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on 
specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, we select a numeric field (e.g. 'Age') and group by 
a categorical field (e.g. 'Sex'). All references use their `@id` values.

In [ ]:
# Choose first record set for EDA
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Find numeric and categorical fields by @id
    rs_obj = None
    for rs in metadata.recordSet:
        if rs['@id'] == record_set_id:
            rs_obj = rs
            break

    numeric_field_id = None
    group_field_id = None
    # Look for fields named 'Age' (numeric) and 'Sex' (categorical)
    for field in rs_obj.get('field', []):
        if field.get('name', '').lower() == 'age':
            numeric_field_id = field['@id']
        if field.get('name', '').lower() in ('sex', 'gender'):
            group_field_id = field['@id']

    # If not found, pick first numeric and first categorical field
    if not numeric_field_id:
        for field in rs_obj.get('field', []):
            if field.get('dataType', '') in ('Integer', 'Float'):
                numeric_field_id = field['@id']
                break
    if not group_field_id:
        for field in rs_obj.get('field', []):
            if field.get('dataType', '') == 'Text':
                group_field_id = field['@id']
                break

    # Print chosen fields
    print(f"Numeric field @id: {numeric_field_id}")
    print(f"Group field @id: {group_field_id}")

    # EDA workflow
    if numeric_field_id and numeric_field_id in df.columns:
        threshold = df[numeric_field_id].median() if df[numeric_field_id].dtype.kind in 'fi' else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by categorical field
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No valid numeric field found for EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric variable (e.g., Age),
and compare across categorical groups (e.g., Sex).

In [ ]:
# Visualization with matplotlib
import matplotlib.pyplot as plt

if record_set_ids and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    df[numeric_field_id].hist(bins=10, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        for group, subset in df.groupby(group_field_id):
            subset[numeric_field_id].hist(alpha=0.5, label=str(group))
        plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.legend()
        plt.show()

## 6. Conclusion
This notebook demonstrates the use of `mlcroissant` to load and process the 
FAIR^2 dataset package defined by a Croissant schema.

- Loaded dataset schema, record sets, fields and columns referenced by `@id`.
- Extracted tabular records and performed basic EDA operations by field IDs.
- Visualized data distributions for key clinical variables.

Further analysis can extend to variable interactions and advanced statistical or ML methods, using the referenced IDs for reproducible workflows.